# Feature engineering


1. **Feature ablation:** drop one feature at a time, compare AUC/deadtime
2. **L1 scan:** sweep `reg_alpha`, watch which feature importances collapse
3. **Permutation importance:** model-agnostic measure of feature contribution at inference

In [ ]:
import sys
sys.path.append("../../../src/ml")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyutils.pyplot import Plot
Plot(verbosity=0)

run = "k"

## Load data

In [ ]:
from assemble import AssembleDataset
asm = AssembleDataset(run=run)
data = asm.assemble_dataset()

all_features = list(data["X_train"].columns)
print(f"Features ({len(all_features)}): {all_features}")

In [ ]:
# Baseline hyperparameters from optimisation
best_hp = {
    "n_estimators": 200,
    "max_depth": 7,
    "learning_rate": 0.1,
    "tree_method": "hist",
    "device": "cuda",
}

## 1. Feature ablation

Drop one feature at a time and compare performance with the baseline (all features).

In [ ]:
from train import Train
from validate import Validate

def train_with_features(data, features, tag, best_hp):
    """Train and validate with a subset of features."""
    subset = {
        "X_train": data["X_train"][features],
        "X_test": data["X_test"][features],
        "X_val": data["X_val"][features],
        "y_train": data["y_train"],
        "y_test": data["y_test"],
        "y_val": data["y_val"],
        "metadata_train": data["metadata_train"],
        "metadata_test": data["metadata_test"],
        "metadata_val": data["metadata_val"],
    }
    trn = Train(subset, run=run, verbosity=0)
    results = trn.train(tag=tag, save_output=False, **best_hp)
    val = Validate(results, run=run, verbosity=0)
    val.roc_auc()
    thr = val.find_threshold(min_eff=0.999, plot=False, show=False)
    return {
        "train_auc": val._train_auc,
        "test_auc": val._test_auc,
        "overfit_gap": val._train_auc - val._test_auc,
        "threshold": thr["threshold"],
        "veto_efficiency": thr["veto_efficiency"],
        "deadtime": thr["deadtime"],
    }

In [ ]:
# Baseline: all features
baseline = train_with_features(data, all_features, "baseline", best_hp)
print(f"Baseline ({len(all_features)} features): test AUC={baseline['test_auc']:.6f}, deadtime={baseline['deadtime']*100:.3f}%")

# Drop one feature at a time
ablation_results = []
for feat in all_features:
    reduced = [f for f in all_features if f != feat]
    result = train_with_features(data, reduced, f"drop_{feat}", best_hp)
    result["dropped"] = feat
    result["n_features"] = len(reduced)
    ablation_results.append(result)
    delta_auc = result["test_auc"] - baseline["test_auc"]
    print(f"  Drop {feat:>10s}: test AUC={result['test_auc']:.6f} (delta={delta_auc:+.6f}), deadtime={result['deadtime']*100:.3f}%")

ablation_df = pd.DataFrame(ablation_results).sort_values("test_auc", ascending=False)
display(ablation_df)

In [ ]:
# Plot ablation results
fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8))

features_sorted = ablation_df["dropped"].values
delta_auc = ablation_df["test_auc"].values - baseline["test_auc"]
delta_deadtime = (ablation_df["deadtime"].values - baseline["deadtime"]) * 100

ax = axes[0]
colours = ["green" if d >= 0 else "red" for d in delta_auc]
ax.barh(features_sorted, delta_auc, color=colours)
ax.axvline(0, color="grey", linestyle="--", alpha=0.5)
ax.set_xlabel("$\Delta$ test AUC (vs baseline)")
ax.set_title("AUC change when feature dropped")

ax = axes[1]
colours = ["green" if d <= 0 else "red" for d in delta_deadtime]
ax.barh(features_sorted, delta_deadtime, color=colours)
ax.axvline(0, color="grey", linestyle="--", alpha=0.5)
ax.set_xlabel("$\Delta$ deadtime [%] (vs baseline)")
ax.set_title("Deadtime change when feature dropped")

plt.tight_layout()
plt.savefig(f"../../../output/images/ml/{run}/features/ablation.png")
plt.show()

## 2. L1 regularisation scan

Sweep `reg_alpha` and track feature importances. Features whose importance drops to ~0 at moderate L1 are candidates for removal.

In [ ]:
import xgboost as xgb

alphas = [0, 0.1, 1, 5, 10, 50, 100]
importance_records = []

for alpha in alphas:
    hp = {**best_hp, "reg_alpha": alpha}
    trn = Train(data, run=run, verbosity=0)
    results = trn.train(tag=f"l1_alpha_{alpha}", save_output=False, **hp)

    # XGBoost feature importance (gain)
    importances = results["model"].get_booster().get_score(importance_type="gain")
    for feat in all_features:
        importance_records.append({
            "reg_alpha": alpha,
            "feature": feat,
            "importance": importances.get(feat, 0.0),
        })

    val = Validate(results, run=run, verbosity=0)
    val.roc_auc()
    print(f"  reg_alpha={alpha:>5}: test AUC={val._test_auc:.6f}")

l1_df = pd.DataFrame(importance_records)

In [ ]:
# Plot feature importance vs L1 strength
fig, ax = plt.subplots(figsize=(8, 5))

for feat in all_features:
    feat_data = l1_df[l1_df["feature"] == feat]
    # Normalise importance relative to alpha=0 value
    base_imp = feat_data[feat_data["reg_alpha"] == 0]["importance"].values[0]
    if base_imp > 0:
        norm_imp = feat_data["importance"].values / base_imp
    else:
        norm_imp = feat_data["importance"].values
    ax.plot(feat_data["reg_alpha"], norm_imp, marker="o", label=feat)

ax.set_xlabel(r"$\alpha$ (L1 regularisation)")
ax.set_xscale("symlog", linthresh=0.1)
ax.set_ylabel("Normalised importance (relative to $\\alpha=0$)")
ax.set_title("Feature importance vs L1 regularisation strength")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
ax.axhline(0, color="grey", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(f"../../../output/images/ml/{run}/features/l1_importance.png")
plt.show()

## 3. Permutation importance

Measures actual impact of each feature on test set predictions by shuffling one feature at a time. Model-agnostic and accounts for feature interactions.

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

# Train baseline model
trn = Train(data, run=run, verbosity=0)
results = trn.train(tag="perm_baseline", save_output=False, **best_hp)

# Permutation importance on test set
perm_result = permutation_importance(
    results["model"],
    data["X_test"], data["y_test"],
    scoring="roc_auc",
    n_repeats=10,
    random_state=42,
)

perm_df = pd.DataFrame({
    "feature": all_features,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std,
}).sort_values("importance_mean", ascending=True)

display(perm_df.sort_values("importance_mean", ascending=False))

In [ ]:
# Plot permutation importance
fig, ax = plt.subplots(figsize=(6.4, 4.8))
ax.barh(perm_df["feature"], perm_df["importance_mean"],
        xerr=perm_df["importance_std"], capsize=3)
ax.set_xlabel("Permutation importance ($\Delta$ AUC)")
ax.set_title("Permutation importance (test set)")

plt.tight_layout()
plt.savefig(f"../../../output/images/ml/{run}/features/permutation_importance.png")
plt.show()

## 4. Summary

Compare all three methods to identify features that are consistently unimportant.

In [ ]:
# Combine results from all three methods
summary = pd.DataFrame({"feature": all_features})

# Ablation: AUC change when dropped (negative = feature is useful)
ablation_map = ablation_df.set_index("dropped")
summary["ablation_delta_auc"] = summary["feature"].map(
    lambda f: ablation_map.loc[f, "test_auc"] - baseline["test_auc"]
)

# L1: normalised importance at reg_alpha=10 (features that survive strong L1 are important)
l1_alpha10 = l1_df[l1_df["reg_alpha"] == 10].set_index("feature")
l1_alpha0 = l1_df[l1_df["reg_alpha"] == 0].set_index("feature")
summary["l1_retained"] = summary["feature"].map(
    lambda f: l1_alpha10.loc[f, "importance"] / l1_alpha0.loc[f, "importance"]
    if l1_alpha0.loc[f, "importance"] > 0 else 0
)

# Permutation importance
perm_map = perm_df.set_index("feature")
summary["perm_importance"] = summary["feature"].map(
    lambda f: perm_map.loc[f, "importance_mean"]
)

summary = summary.sort_values("perm_importance", ascending=False)
display(summary)